# 14.6 輸入輸出權重應該共享嗎？


模型讀到「貓」時，要從詞表查出代表貓的向量；準備輸出下一字時，又要用當前向量替「貓、狗、車」等候選打分。這兩件事都需要一張「每個token對應一列數字」的表，能否只存一張？把[輸入查表](https://birdhackor.github.io/tiny-perceptron-vlm/2.1.html)和[輸出打分](https://birdhackor.github.io/tiny-perceptron-vlm/4.6.html)放在一起看：token是文字切分後的單位，詞表是所有可用單位的清單；輸入取一列，輸出拿當前向量與各列做點積。

假設詞表只有三項、每項用兩個特徵，輸入表E是三列兩欄，共六個權重。若輸出再存一張同形狀的表，就多六個。Weight tying（權重共享）把輸出層的權重設成同一張E，輸出分數就是 `h @ E.T`，h是當前兩個特徵；`@`是矩陣乘法，`.T`將E的列與欄對調，讓h一次與各候選做點積。點積就是同位置的數字先相乘，再把乘積相加：若E三列為 `[1,0]`、`[0,1]`、`[1,1]`，h為 `[2,1]`，三個分數分別是 `2×1+1×0=2`、`2×0+1×1=1`、`2×1+1×1=3`。`h @ E.T` 是一次完成這三列同樣的運算，並沒有多一條打分規則。這個點積仍是分數，還沒有轉成機率。

關鍵在於「同一張」。複製E的初值到另一張表，只是此刻數字相同；日後兩張表收到不同學習訊號，仍會各自改變。共享則是兩個模組引用同一個Parameter，Parameter是PyTorch用來記錄可學數字的物件，因此更新時只更新這一份。下面刻意建立共享、複製兩種輸出，直接觀察差別。


In [1]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [2]:
import torch
from torch import nn

embedding = nn.Embedding(3, 2)
with torch.no_grad():
    embedding.weight.copy_(torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]))
tied = nn.Linear(2, 3, bias=False)
tied.weight = embedding.weight
copied = nn.Linear(2, 3, bias=False)
with torch.no_grad():
    copied.weight.copy_(embedding.weight)
h = torch.tensor([[2.0, 1.0]])
print("原分數", tied(h).tolist(), copied(h).tolist())
print("同一物件", tied.weight is embedding.weight, copied.weight is embedding.weight)
with torch.no_grad():
    embedding.weight[0, 0] += 1
print("改表後", tied(h).tolist(), copied(h).tolist())

原分數 [[2.0, 1.0, 3.0]] [[2.0, 1.0, 3.0]]
同一物件 True False
改表後 [[4.0, 1.0, 3.0]] [[2.0, 1.0, 3.0]]


輸入h是一列兩欄，`nn.Linear(2,3)`將它轉成三個候選分數。`bias=False`省去額外偏移，使結果只看權重表。`no_grad`暫停記錄學習用的導數，讓我們直接設初值和手動改數字。第一行兩者都是 `[[2,1,3]]`，第二行是 `True False`。修改第一列第一項之後，共享輸出變成 `[[4,1,3]]`，複製輸出仍是 `[[2,1,3]]`，因為只有共享版本真的讀到被改的同一份數字。

真實模型若詞表大小V、特徵寬度D，共享可省 `V×D` 個權重；但輸入表和輸出表也失去各自調整的自由度。這是模型設計選擇，不能單憑省參數斷言品質更好。模型儲存、載入或量化（改用較少位元表示權重）後，也應再次核對共享關係，以免轉換程序建立了兩份表。

練習只把 `embedding.weight[0,0] += 1` 改成第二列第二項加1。先用h第二項是1預測共享的第二個分數只增加1，再執行核對。若只看兩個表最初數字相等，會漏掉這節最重要的物件共享差別。

<details>
<summary>選讀：來源、量測條件與原始實報</summary>

[Press與Wolf的原論文第3節](https://arxiv.org/pdf/1608.05859v3)說明輸入與輸出表的綁定。我們在[T.8](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.8)保存的共享組有124,672個參數，比基準少264×64=16,896個。240次更新後，驗證代價2.36276、最後檢查2.38422，這輪高於基準2.26331與2.29163。

還要看起點：共享組把原本獨立的輸出表改成embedding那份初值，訓練前代價43.87647，基準只有5.76007。共有形狀的其他表雖然沿用同一初值，輸出分布已不同，所以不能用這次短訓斷言共享本身傷害品質。省下的權重數是可核對的結構事實；需要多少訓練適應、最後品質如何，仍是另一個實驗問題。

</details>
